<a href="https://colab.research.google.com/github/tiwimaniku8-design/deteksi-spam-sms/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/tiwimaniku8-design/deteksi-spam-sms/main/data/raw/SMSSpamCollection"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
df = df.drop_duplicates().reset_index(drop=True)

print(df.shape)

(5169, 2)


In [4]:
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print("NLTK siap")

NLTK siap


In [5]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text, use_stem=False):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' urlkhusus ', text)
    text = re.sub(r'\b\d{8,}\b', ' nomorkhusus ', text)
    text = re.sub(r'£|\bpounds?\b', ' uangkhusus ', text)

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]

    if use_stem:
        tokens = [stemmer.stem(t) for t in tokens]

    return ' '.join(tokens)

df['bersih'] = df['message'].apply(preprocess)

for i in range(3):
    print("Asli  :", df['message'][i])
    print("Bersih:", df['bersih'][i])
    print()

print("urlkhusus   :", df['bersih'].str.contains('urlkhusus').sum())
print("nomorkhusus :", df['bersih'].str.contains('nomorkhusus').sum())
print("uangkhusus  :", df['bersih'].str.contains('uangkhusus').sum())

Asli  : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Bersih: go jurong point crazy available bugis n great world la e buffet cine got amore wat

Asli  : Ok lar... Joking wif u oni...
Bersih: ok lar joking wif u oni

Asli  : Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Bersih: free entry wkly comp win fa cup final tkts may text fa receive entry question std txt rate c apply

urlkhusus   : 91
nomorkhusus : 333
uangkhusus  : 240


In [6]:
print("Pesan asli berisi http/www:", df['message'].str.contains(r'https?://|www\.', case=False, regex=True).sum())
print("Pesan asli berisi £:", df['message'].str.contains('£', regex=False).sum())

Pesan asli berisi http/www: 91
Pesan asli berisi £: 218


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, recall_score, precision_score

df['bersih_stem'] = df['message'].apply(lambda t: preprocess(t, use_stem=True))

def evaluasi(kolom):
    X_train, X_test, y_train, y_test = train_test_split(
        df[kolom], df['label'], test_size=0.2, random_state=42, stratify=df['label']
    )
    tfidf = TfidfVectorizer()
    X_train_t = tfidf.fit_transform(X_train)  # fit hanya pada data latih
    X_test_t = tfidf.transform(X_test)

    model = MultinomialNB()
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)

    return {
        'precision_spam': round(precision_score(y_test, y_pred, pos_label='spam'), 4),
        'recall_spam': round(recall_score(y_test, y_pred, pos_label='spam'), 4),
        'f1_spam': round(f1_score(y_test, y_pred, pos_label='spam'), 4),
    }

print("Tanpa stemming:", evaluasi('bersih'))
print("Dengan stemming:", evaluasi('bersih_stem'))

Tanpa stemming: {'precision_spam': 1.0, 'recall_spam': 0.7481, 'f1_spam': 0.8559}
Dengan stemming: {'precision_spam': 1.0, 'recall_spam': 0.7634, 'f1_spam': 0.8658}


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for kolom in ['bersih', 'bersih_stem']:
    pipeline = make_pipeline(TfidfVectorizer(), MultinomialNB())
    skor = cross_val_score(pipeline, df[kolom], df['label'], cv=cv, scoring='f1_macro')
    print(kolom, "F1 macro rata-rata:", round(skor.mean(), 4), "±", round(skor.std(), 4))

bersih F1 macro rata-rata: 0.9275 ± 0.0066
bersih_stem F1 macro rata-rata: 0.9243 ± 0.0133
